In [ ]:
import psycopg2
import os
import pandas as pd
import yaml

In [10]:
os.environ["PGSERVICEFILE"] = os.path.join(os.getcwd(), ".pg_service.conf")
os.environ["PGPASSFILE"] = os.path.join(os.getcwd(), ".pgpass")

with open(os.environ["PGPASSFILE"]) as f:
    pgpass = f.read().strip()

In [11]:
beaver = psycopg2.connect(service="beam_neb0",password=pgpass)
cursor = beaver.cursor()

In [ ]:
## schema
cursor.execute("""
    select
        c.table_schema,
        c.table_name,
        c.column_name,
        c.data_type,
        c.is_nullable,
        c.column_default,
        pgd.description AS column_comment
    from information_schema.columns c
    left join pg_catalog.pg_statio_all_tables st
        ON st.schemaname = c.table_schema
        AND st.relname = c.table_name
    left join pg_catalog.pg_description pgd
        ON pgd.objoid = st.relid
        AND pgd.objsubid = c.ordinal_position
    where c.table_schema = 'public'
    order by c.table_schema, c.table_name, c.ordinal_position;
""")

columns_df = pd.DataFrame(cursor.fetchall(), columns=[
    "schema", "table", "column", "data_type", "nullable", "default", "comment"])